<a href="https://colab.research.google.com/github/akshin-kuvshin/numerical-methods/blob/main/lab6/lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №6. Дифференцирование по равноотстоящим узлам

## Выполнил: Данила Аксёнов, гр. 932420

## Вариант: 1

### P.S.: часть кода взята из лабораторной работы №5, как того и требует формулировка задания: "Используя результаты задания 10...".

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
print('Setup completed')

Setup completed


### Исходные данные:

In [ ]:
n = 11
x = np.linspace(1, 2, num=n)
h = 0.1 # шаг таблицы
y = np.array([0.322, 0.284, 0.241, 0.193, 0.135, 0.063, -0.031, -0.164, -0.369, -0.741, -1.664])
eps = 0.0005 # общая абсолютная погрешность

In [ ]:
# точки, значения функции в которых необходимо аппроксимировать
x1_star = 0.98
x2_star = 1.56

In [ ]:
d = [y] # таблица конечных разностей
for i in range(1, n):
    d_i = np.zeros(n - i)
    for j in range(0, n - i):
        d_i[j] = d[i - 1][j + 1] - d[i - 1][j]
    d.append(d_i)
d

[array([ 0.322,  0.284,  0.241,  0.193,  0.135,  0.063, -0.031, -0.164,
        -0.369, -0.741, -1.664]),
 array([-0.038, -0.043, -0.048, -0.058, -0.072, -0.094, -0.133, -0.205,
        -0.372, -0.923]),
 array([-0.005, -0.005, -0.01 , -0.014, -0.022, -0.039, -0.072, -0.167,
        -0.551]),
 array([-5.55111512e-17, -5.00000000e-03, -4.00000000e-03, -8.00000000e-03,
        -1.70000000e-02, -3.30000000e-02, -9.50000000e-02, -3.84000000e-01]),
 array([-0.005,  0.001, -0.004, -0.009, -0.016, -0.062, -0.289]),
 array([ 0.006, -0.005, -0.005, -0.007, -0.046, -0.227]),
 array([-1.10000000e-02, -8.32667268e-17, -2.00000000e-03, -3.90000000e-02,
        -1.81000000e-01]),
 array([ 0.011, -0.002, -0.037, -0.142]),
 array([-0.013, -0.035, -0.105]),
 array([-0.022, -0.07 ]),
 array([-0.048])]

---

In [ ]:
# ищем максимальный порядок правильных конечных разностей
for i in range(n - 1):
    if np.abs(d[i]).max() >= (2 ** i) * eps and np.abs(d[i + 1]).max() < (2 ** (i + 1)) * eps:
        r = i
        break
r

7

In [ ]:
r = min(r, 4) # в дальнейшем будем использовать правильные конечные разности не выше четвёртого порядка
r

4

In [ ]:
r = 5 # конечные разности пятого порядка, пусть они и не являются "правильными", понадобятся нам на этапе вычисления погрешностей метода

---

In [ ]:
# предпосчитаем факториалы для чисел до r=4 включительно - это пригодится нам в дальнейшем
fact = np.ones(r + 1)
for i in range(1, r + 1):
    fact[i] = fact[i - 1] * i
fact

array([  1.,   1.,   2.,   6.,  24., 120.])

## **Задание 11**

* Вычислить 1 и 2 производные в точках $ x_1^* $ и $ x_2^* $;
* Считая, что заданные значения известны с верными знаками, вычислить следующие погрешности: вычислительную, метода и полную.

### Решение:

Первая точка, значения производных в которой необходимо вычислить $ - $ $ x_1^* = 0.98 $ $ - $, находится вблизи начала таблицы, т.е. вблизи точки $ x_0 = 1 $, поэтому здесь уместно применить _1-ю интерполяционную формулу Ньютона_ (для интерполирования вперёд).

Дифференцируя 1-ю интерполяционную формулу Ньютона (для интерполирования вперёд) и оставляя только слагаемые с конечными разностями не выше четвёртого порядка, получаем приближенное выражение для вычисления первой производной функции $ f $ в точке $ x_1^* $:

$$ f'(x_1^*) = f'(x_0 + t_1^* h) \approx \frac{1}{h} \left( \Delta y_0 + \frac{\Delta^2 y_0}{2!} (2 t_1^* - 1) + \frac{\Delta^3 y_0}{3!} (3 (t_1^*)^2 - 6 t_1^* + 2) + \frac{\Delta^4 y_0}{4!} (4 (t_1^*)^3 - 18 (t_1^*)^2 + 22 t_1^* - 6) \right) $$

(см. формулу 3.24 из учебника).

Дифференцируя полученное выражение ещё раз, получим выражение для вычисления второй производной функции $ f $ в точке $ x_1^* $:

$$ f''(x_1^*) =  f''(x_0 + t_1^* h) \approx \frac{1}{h^2} \left( \Delta^2 y_0 + \Delta^3 y_0 (t_1^* - 1) + \Delta^4 y_0 \left( \frac{(t_1^*)^2}{2} - \frac{3 t_1^*}{2} + \frac{11}{12} \right) \right). $$

In [ ]:
t1_star = (x1_star - x[0]) / h
f_deriv1_x1_star = (d[1][0] \
                 +  d[2][0] / fact[2] * (2 * t1_star - 1) \
                 +  d[3][0] / fact[3] * (3 * t1_star ** 2 - 6 * t1_star + 2) \
                 +  d[4][0] / fact[4] * (4 * t1_star ** 3 - 18 * t1_star ** 2 + 22 * t1_star - 6)) / h
f_deriv2_x1_star = (d[2][0] \
                 +  d[3][0] * (t1_star - 1) \
                 +  d[4][0] * (t1_star ** 2 / 2 - 3 * t1_star / 2 + 11 / 12)) / h ** 2
f_deriv1_x1_star, f_deriv2_x1_star

(np.float64(-0.32176666666666787), np.float64(-1.118333333333315))

Получили следующие результаты: $ \quad \mathbf{f'(x_1^*) = f'(0.98) \approx -0.3217(6)} \quad $ и $ \quad \mathbf{f''(x_1^*) = f''(0.98) \approx -1.118(3)} $.

Оценим теперь неустранимую (вычислительную) погрешность ($ \Delta_Н $) и погрешность метода ($ \Delta_М $), чтобы найти полную погрешность, как их сумму ($ \Delta_П = \Delta_Н + \Delta_М $). Для $ f'(x_1^*) $ обозначим эти погрешности $ \Delta'_{Н1} $, $ \Delta'_{М1} $ и $ \Delta'_{П1} $ соответственно; для $ f''(x_1^*) $ $ - $ $ \Delta''_{Н1} $, $ \Delta''_{М1} $ и $ \Delta''_{П1} $ соответственно.

1. Неустранимая погрешность получается _из расчётной формулы_ взятием по модулю и заменой $ \Delta^m y_i $ на $ 2^m \epsilon $, где $ \epsilon $ $ - $ абсолютная погрешность, одинаковая для всех $ y_i $. Таким образом:

$$ \Delta'_{Н1} = \frac{1}{h} \left( 2 \epsilon + \frac{2^2 \epsilon}{2!} \lvert 2 t_1^* - 1 \rvert + \frac{2^3 \epsilon}{3!} \lvert 3 (t_1^*)^2 - 6 t_1^* + 2 \rvert + \frac{2^4 \epsilon}{4!} \lvert 4 (t_1^*)^3 - 18 (t_1^*)^2 + 22 t_1^* - 6 \rvert \right), $$

$$ \Delta''_{Н1} = \frac{1}{h^2} \left( 2^2 \epsilon + 2^3 \epsilon \lvert t_1^* - 1 \rvert + 2^4 \epsilon \left\lvert \frac{(t_1^*)^2}{2} - \frac{3 t_1^*}{2} + \frac{11}{12} \right\rvert \right); $$

2. Погрешность метода получается _из остаточного члена_ взятием по модулю и заменой $ f^{(m)}(\xi) $ на $ M_m = \max_{x \in [a; b]} \lvert f^{(m)}(x) \rvert \approx \max_i \frac{\lvert \Delta^m y_i \rvert}{h^m} $. Сам же остаточный член получается _из расчётной формулы_ взятием последнего отброшенного слагаемого и заменой $ \Delta^m y_i $ на $ h^m f^{(m)}(\xi) $. Остаточные члены расчётных формул для $ f'(x_1^*) $ и $ f''(x_1^*) $ обозначим $ R'_4(x_1^*) $ и $ R''_4(x_1^*) $ соответственно. Тогда:

$$ R'_4(x_1^*) = R'_4(x_0 + t_1^* h) = \frac{1}{h} h^5 f^{(5)}(\xi) \frac{5 (t_1^*)^4 - 40 (t_1^*)^3 + 105 (t_1^*)^2 - 100 t_1^* + 24}{5!}, $$

$$ R''_4(x_1^*) = R''_4(x_0 + t_1^* h) = \frac{1}{h^2} h^5 f^{(5)}(\xi) \frac{20 (t_1^*)^3 - 120 (t_1^*)^2 + 210 t_1^* - 100}{5!}; $$

$$ \Downarrow $$

$$ \Delta'_{М1} \approx \frac{1}{h} \max_i \lvert \Delta^5 y_i \rvert \frac{\lvert 5 (t_1^*)^4 - 40 (t_1^*)^3 + 105 (t_1^*)^2 - 100 t_1^* + 24 \rvert}{5!}, $$

$$ \Delta''_{М1} \approx \frac{1}{h^2} \max_i \lvert \Delta^5 y_i \rvert \frac{\lvert 20 (t_1^*)^3 - 120 (t_1^*)^2 + 210 t_1^* - 100 \rvert}{5!}. $$

In [ ]:
calc_error_f_deriv1_x1_star = (2 * eps \
                            +  2 ** 2 * eps / fact[2] * abs(2 * t1_star - 1) \
                            +  2 ** 3 * eps / fact[3] * abs(3 * t1_star ** 2 - 6 * t1_star + 2) \
                            +  2 ** 4 * eps / fact[4] * abs(4 * t1_star ** 3 - 18 * t1_star ** 2 + 22 * t1_star - 6)) / h
calc_error_f_deriv2_x1_star = (2 ** 2 * eps \
                            +  2 ** 3 * eps * abs(t1_star - 1) \
                            +  2 ** 4 * eps * abs(t1_star ** 2 / 2 - 3 * t1_star / 2 + 11 / 12)) / h ** 2

d_5_abs_max = np.abs(d[5]).max()
method_error_f_deriv1_x1_star = d_5_abs_max * abs(5 * t1_star ** 4 - 40 * t1_star ** 3 + 105 * t1_star ** 2 - 100 * t1_star + 24) / fact[5] / h
method_error_f_deriv2_x1_star = d_5_abs_max * abs(20 * t1_star ** 3 - 120 * t1_star ** 2 + 210 * t1_star - 100) / fact[5] / h ** 2

full_error_f_deriv1_x1_star = calc_error_f_deriv1_x1_star + method_error_f_deriv1_x1_star
full_error_f_deriv2_x1_star = calc_error_f_deriv2_x1_star + method_error_f_deriv2_x1_star

In [ ]:
calc_error_f_deriv1_x1_star, method_error_f_deriv1_x1_star, full_error_f_deriv1_x1_star

(np.float64(0.08330666666666668),
 np.float64(0.9179879999999998),
 np.float64(1.0012946666666664))

In [ ]:
calc_error_f_deriv2_x1_star, method_error_f_deriv2_x1_star, full_error_f_deriv2_x1_star

(np.float64(1.6693333333333331),
 np.float64(27.79993333333331),
 np.float64(29.469266666666645))

Были получены следующие результаты:

$$ \mathbf{\Delta'_{Н1} = 0.08330(6)}, \quad \mathbf{\Delta'_{М1} \approx 0.917988}, \quad \mathbf{\Delta'_{П1} = \Delta'_{Н1} + \Delta'_{М1} \approx 1.001294(6)}, $$

$$ \mathbf{\Delta''_{Н1} = 1.669(3)}, \quad \mathbf{\Delta''_{М1} \approx 27.7999(3)}, \quad \mathbf{\Delta''_{П1} = \Delta''_{Н1} + \Delta''_{М1} \approx 29.4692(6)}. $$

**Огромные** значения $ \Delta''_{М1} $ и, соответственно, $ \Delta''_{П1} $ можно объяснить так: _точка $ x_1^* = 0.98 $ лежит левее точки $ x_0 = 1 \quad \Rightarrow \quad t_1^* = \frac{x_0 - x_1^*}{h} < 0 $, и имеет место не интерполяция, а **экстраполяция**, что вкупе с крупным шагом $ h = 0.1 $ и высоким порядком производной даёт высокие неустранимую погрешность и погрешность метода, причём последняя значительно доминирует_.

---

Вторая точка $ x_2^* = 1.56 $ находится в середине таблицы, причём расстояние до ближайшего узла справа $ - $ $ x_6 = 1.6 $ $ - $ немного меньше, чем аналогичное расстояние слева (до узла $ x_5 = 1.5 $), поэтому я решил использовать _2-ю формулу Гаусса_ (для интерполирования назад).

* Используя для расчёта конечные разности не выше 4-ого порядка, а для вычисления погрешностей метода $ - $ разности 5-ого порядка,
* введя обозначения, аналогичные предыдущим,
* дважды дифференцируя 2-ю формулу Гаусса (для интерполирования назад) и её остаточный член для чётной степени
* и используя вышеописанные приёмы,

получаем следующие выражения:

$$ f'(x_2^*) = f'(x_6 + t_2^* h) \approx \frac{1}{h} \left( \Delta y_5 + \frac{\Delta^2 y_5}{2!} (2 t_2^* + 1) + \frac{\Delta^3 y_4}{3!} (3 (t_2^*)^2 - 1) + \frac{\Delta^4 y_4}{4!} (4 (t_2^*)^3 + 6 (t_2^*)^2 - 2 t_2^* - 2) \right), $$

$$ f''(x_2^*) =  f''(x_6 + t_2^* h) \approx \frac{1}{h^2} \left( \Delta^2 y_5 + \Delta^3 y_4 t_2^* + \Delta^4 y_4 \left( \frac{(t_2^*)^2}{2} + \frac{t_2^*}{2} - \frac{1}{12} \right) \right); $$

---

$$ \Delta'_{Н2} = \frac{1}{h} \left( 2 \epsilon + \frac{2^2 \epsilon}{2!} \lvert 2 t_2^* + 1 \rvert + \frac{2^3 \epsilon}{3!} \lvert 3 (t_2^*)^2 - 1 \rvert + \frac{2^4 \epsilon}{4!} \lvert 4 (t_2^*)^3 + 6 (t_2^*)^2 - 2 t_2^* - 2 \rvert \right), $$

$$ \Delta''_{Н2} = \frac{1}{h^2} \left( 2^2 \epsilon + 2^3 \epsilon \lvert t_2^* \rvert + 2^4 \epsilon \left\lvert \frac{(t_2^*)^2}{2} + \frac{t_2^*}{2} - \frac{1}{12} \right\rvert \right); $$

---

$$ \bar{R}'_4(x_2^*) = \bar{R}'_4(x_6 + t_2^* h) = \frac{1}{h} h^5 f^{(5)}(\xi) \frac{5 (t_2^*)^4 - 15 (t_2^*)^2 + 4}{5!}, $$

$$ \bar{R}''_4(x_2^*) = \bar{R}''_4(x_6 + t_2^* h) = \frac{1}{h^2} h^5 f^{(5)}(\xi) \frac{20 (t_2^*)^3 - 30 t_2^*}{5!}; $$

$$ \Downarrow $$

$$ \Delta'_{М2} \approx \frac{1}{h} \max_i \lvert \Delta^5 y_i \rvert \frac{\lvert 5 (t_2^*)^4 - 15 (t_2^*)^2 + 4 \rvert}{5!}, $$

$$ \Delta''_{М2} \approx \frac{1}{h^2} \max_i \lvert \Delta^5 y_i \rvert \frac{\lvert 20 (t_2^*)^3 - 30 t_2^* \rvert}{5!}. $$

In [ ]:
t2_star = (x2_star - x[6]) / h
f_deriv1_x2_star = (d[1][5] \
                 +  d[2][5] / fact[2] * (2 * t2_star + 1) \
                 +  d[3][4] / fact[3] * (3 * t2_star ** 2 - 1) \
                 +  d[4][4] / fact[4] * (4 * t2_star ** 3 + 6 * t2_star ** 2 - 2 * t2_star - 2)) / h
f_deriv2_x2_star = (d[2][5] \
                 +  d[3][4] * t2_star \
                 +  d[4][4] * (t2_star ** 2 / 2 + t2_star / 2 - 1 / 12)) / h ** 2
f_deriv1_x2_star, f_deriv2_x2_star

(np.float64(-0.9609599999999999), np.float64(-2.894666666666666))

In [ ]:
calc_error_f_deriv1_x2_star = (2 * eps \
                            +  2 ** 2 * eps / fact[2] * abs(2 * t2_star + 1) \
                            +  2 ** 3 * eps / fact[3] * abs(3 * t2_star ** 2 - 1) \
                            +  2 ** 4 * eps / fact[4] * abs(4 * t2_star ** 3 + 6 * t2_star ** 2 - 2 * t2_star - 2)) / h
calc_error_f_deriv2_x2_star = (2 ** 2 * eps \
                            +  2 ** 3 * eps * abs(t2_star) \
                            +  2 ** 4 * eps * abs(t2_star ** 2 / 2 + t2_star / 2 - 1 / 12)) / h ** 2

# d_5_abs_max вычислено ранее
method_error_f_deriv1_x2_star = d_5_abs_max * abs(5 * t2_star ** 4 - 15 * t2_star ** 2 + 4) / fact[5] / h
method_error_f_deriv2_x2_star = d_5_abs_max * abs(20 * t2_star ** 3 - 30 * t2_star) / fact[5] / h ** 2

full_error_f_deriv1_x2_star = calc_error_f_deriv1_x2_star + method_error_f_deriv1_x2_star
full_error_f_deriv2_x2_star = calc_error_f_deriv2_x2_star + method_error_f_deriv2_x2_star

In [ ]:
calc_error_f_deriv1_x2_star, method_error_f_deriv1_x2_star, full_error_f_deriv1_x2_star

(np.float64(0.01711999999999998),
 np.float64(0.0326879999999999),
 np.float64(0.04980799999999988))

In [ ]:
calc_error_f_deriv2_x2_star, method_error_f_deriv2_x2_star, full_error_f_deriv2_x2_star

(np.float64(0.5226666666666667),
 np.float64(2.027866666666666),
 np.float64(2.5505333333333327))

Получили следующие результаты:

$$ \mathbf{f'(x_2^*) = f'(1.56) \approx -0.96096}, \quad \mathbf{f''(x_2^*) = f''(1.56) \approx -2.894(6)}, $$

$$ \mathbf{\Delta'_{Н2} = 0.01712}, \quad \mathbf{\Delta'_{М2} \approx 0.032688}, \quad \mathbf{\Delta'_{П2} = \Delta'_{Н2} + \Delta'_{М2} \approx 0.049808}, $$

$$ \mathbf{\Delta''_{Н2} = 0.522(6)}, \quad \mathbf{\Delta''_{М2} \approx 2.0278(6)}, \quad \mathbf{\Delta''_{П2} = \Delta''_{Н2} + \Delta''_{М2} \approx 2.5505(3)}. $$

В этот раз точность гораздо выше $ - $ **все погрешности в разы меньше**, что обусловлено тем, что _теперь мы **интерполируем** непосредственно **в середине таблицы**, где есть много узлов как слева, так и справа $ \Rightarrow $ конечные разности не "раздуты" краевыми эффектами_.